# YOLOv1 Training v2 — Official Darknet ImageNet-1K Backbone

**Why v2?** The original training (`YOLO_train.ipynb`) used a backbone pretrained on
ImageNet10 (10 classes, ~13K images). The result was a complete class-head collapse:
100% of detections were predicted as "person" and mAP@0.5 = 0.07%.

The YOLO paper (Redmon et al., §2.2) states: *"We pretrain our convolutional layers on
the ImageNet 1000-class competition dataset … for approximately a week, achieving top-5
accuracy of 88%."* This notebook replaces the ImageNet10 backbone with the official
pjreddie Darknet weights trained on full ImageNet-1K, then re-runs the full 60-epoch
VOC 2012 detection training.

**New cells (1–4 below):** download + convert official weights → npz, verify backbone,
insert into YOLO.  
**Unchanged:** all hyperparameters, LR schedule, loss, augmentation, and training loop
are identical to `YOLO_train.ipynb`.

In [ ]:
!nvidia-smi

In [ ]:
!rm -rf /content/yolov1-cupy
!git clone https://github.com/mihnea-popescu/yolov1-cupy.git

import sys
sys.path.append('/content/yolov1-cupy')

from main import TestClass
test = TestClass()
test.test()

In [ ]:
# Pascal VOC 2012 dataset download + extract (Kaggle)
!curl -L -o /content/pascal-voc-2012-dataset.zip https://www.kaggle.com/api/v1/datasets/download/gopalbhattrai/pascal-voc-2012-dataset
print("Extracting Pascal VOC dataset (quiet unzip)...")
!unzip -q /content/pascal-voc-2012-dataset.zip -d /content/yolov1-cupy
!rm /content/pascal-voc-2012-dataset.zip
print("Pascal VOC dataset ready under /content/yolov1-cupy (VOC2012_train_val / VOC2012_test)")

In [ ]:
# Mount Google Drive.  v2 checkpoints go to a separate subdirectory so they
# don't overwrite the original run.
from google.colab import drive
drive.mount('/content/drive')

import os, shutil

PATH_PREFIX    = "/content/drive/MyDrive/GWU_Machine-Learning-Final-Project/v2"
DRIVE_CKPT_DIR = f"{PATH_PREFIX}/checkpoints"
LOCAL_CKPT_DIR = "/content/yolov1-cupy/models"
LOCAL_LOG_PATH = "/content/yolov1-cupy/models/yolo_training_logs_v2.log"
DRIVE_LOG_PATH = f"{DRIVE_CKPT_DIR}/yolo_training_logs_v2.log"

os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
os.makedirs(LOCAL_CKPT_DIR, exist_ok=True)
print("Drive checkpoints dir :", DRIVE_CKPT_DIR)
print("Local  checkpoints dir:", LOCAL_CKPT_DIR)

## Step 1 — Download official Darknet backbone weights

The `darknet.weights` file from pjreddie.com is the reference Darknet backbone
(20 conv layers) trained on ImageNet-1000.  This is exactly the backbone described
in the YOLO paper.

In [ ]:
DARKNET_WEIGHTS_PATH = "/content/darknet.weights"

!wget -q --show-progress -O {DARKNET_WEIGHTS_PATH} https://pjreddie.com/media/files/darknet.weights

import os
size_mb = os.path.getsize(DARKNET_WEIGHTS_PATH) / 2**20
print(f"Downloaded: {DARKNET_WEIGHTS_PATH}  ({size_mb:.1f} MB)")
assert size_mb > 50, "File too small — download likely failed."

## Step 2 — Convert binary `.weights` → our `.npz` format

The Darknet binary format stores, for each Conv+BN block in network order:
1. BN biases (β)
2. BN scales (γ)
3. BN running mean
4. BN running variance
5. Conv weights (NCHW)

The header is either 4 × `int32` (old Darknet) or 3 × `int32` + 1 × `uint64`
(Darknet ≥ 2.0).  The script auto-detects both formats by comparing the actual
file size against the expected size for each combination.

In [ ]:
import struct
import numpy as np
import cupy as cp

from darknet import Darknet
from conv2d import Conv2D
from batchnorm2d import BatchNorm2D

OUTPUT_NPZ = "/content/yolov1-cupy/darknet_pretrained_imagenet1k.npz"

darknet = Darknet(num_classes=1000)

# Collect (Conv2D, BatchNorm2D) pairs in the order they appear in darknet.layers.
# Each Conv2D is immediately followed by its BatchNorm2D in the flat layer list.
pairs = []
pending_conv = None
for layer in darknet.layers:
    if isinstance(layer, Conv2D):
        pending_conv = layer
    elif isinstance(layer, BatchNorm2D):
        assert pending_conv is not None, "BatchNorm2D without preceding Conv2D"
        pairs.append((pending_conv, layer))
        pending_conv = None
assert len(pairs) == 20, f"Expected 20 conv+bn pairs, got {len(pairs)}"
print(f"Found {len(pairs)} Conv+BN pairs in darknet.layers")


def _block_bytes(conv, with_bn):
    """Byte count for one conv block in the .weights file."""
    out_ch = int(conv.weights.shape[0])
    n_conv = int(np.prod(conv.weights.shape))
    if with_bn:
        return (out_ch * 4 + n_conv) * 4   # 4 BN arrays of out_ch + conv weights
    else:
        return (out_ch + n_conv) * 4        # conv bias + conv weights


def _expected_size(header_ints, with_bn):
    size = header_ints * 4
    for conv, _ in pairs:
        size += _block_bytes(conv, with_bn)
    size += (1000 + 1000 * 1024) * 4   # FC head: 1000 biases + 1000×1024 weights
    return size


actual = os.path.getsize(DARKNET_WEIGHTS_PATH)
print(f"\nFile size: {actual:,} bytes ({actual / 2**20:.2f} MB)")
print(f"{'header':>8s}  {'BN':>4s}  {'expected':>16s}  match")
print("-" * 48)

best = None
for h in (4, 5):
    for bn in (True, False):
        exp = _expected_size(h, bn)
        diff = actual - exp
        match = "MATCH" if diff == 0 else f"diff={diff:+,}"
        print(f"{h:>8d}  {'yes' if bn else 'no ':>4s}  {exp:>16,}  {match}")
        if diff == 0 and best is None:
            best = (h, bn)

if best is None:
    # None matched exactly — pick closest and warn
    candidates = [(abs(actual - _expected_size(h, bn)), h, bn) for h in (4, 5) for bn in (True, False)]
    candidates.sort()
    _, hdr, has_bn = candidates[0]
    print(f"\nNo exact match. Using closest: header={hdr} int32, BN={'yes' if has_bn else 'no'}")
    print("Conversion will continue but may fail with a shape error.")
else:
    hdr, has_bn = best
    print(f"\nDetected format: header={hdr} int32, BN={'yes' if has_bn else 'no'}")

In [ ]:
# Read weights from the binary file and populate darknet.layers in-place.
with open(DARKNET_WEIGHTS_PATH, "rb") as f:
    # Header
    raw_hdr = f.read(hdr * 4)
    header  = struct.unpack(f"{hdr}i", raw_hdr)
    print(f"Header ({hdr} int32s): {header}")

    # Backbone: 20 Conv+BN blocks
    for i, (conv, bn) in enumerate(pairs):
        out_ch = int(conv.weights.shape[0])
        shape  = conv.weights.shape  # (out_ch, in_ch, kH, kW)

        if has_bn:
            beta         = np.frombuffer(f.read(out_ch * 4), dtype=np.float32).copy()
            gamma        = np.frombuffer(f.read(out_ch * 4), dtype=np.float32).copy()
            running_mean = np.frombuffer(f.read(out_ch * 4), dtype=np.float32).copy()
            running_var  = np.frombuffer(f.read(out_ch * 4), dtype=np.float32).copy()
            bn.beta         = cp.asarray(beta,         dtype=darknet.dtype)
            bn.gamma        = cp.asarray(gamma,        dtype=darknet.dtype)
            bn.running_mean = cp.asarray(running_mean, dtype=darknet.dtype)
            bn.running_var  = cp.asarray(running_var,  dtype=darknet.dtype)
        else:
            # No BN in file: read and discard conv bias (our conv layers use bias=False)
            _ = np.frombuffer(f.read(out_ch * 4), dtype=np.float32)
            print(f"  conv{i:02d}: no BN in file — keeping default BN init (gamma=1, beta=0)")

        n_w     = int(np.prod(shape))
        weights = np.frombuffer(f.read(n_w * 4), dtype=np.float32).copy().reshape(shape)
        conv.weights = cp.asarray(weights, dtype=darknet.dtype)

        print(
            f"  conv{i:02d} ({int(shape[1])}→{out_ch:4d}, {shape[2]}×{shape[3]}): "
            f"w mean={weights.mean():+.4f}  std={weights.std():.4f}  "
            + (f"bn_gamma mean={float(gamma.mean()):+.4f}" if has_bn else "")
        )

    # Discard the 1000-class FC head (not used for YOLO)
    _ = np.frombuffer(f.read(1000 * 4),        dtype=np.float32)  # biases
    _ = np.frombuffer(f.read(1000 * 1024 * 4), dtype=np.float32)  # weights

    remaining = f.read()
    if remaining:
        print(f"\nWARNING: {len(remaining)} bytes left unread in file.")
    else:
        print("\nFile fully consumed — conversion complete.")

# Save in our npz format
darknet.save_weights(OUTPUT_NPZ)
npz_mb = os.path.getsize(OUTPUT_NPZ) / 2**20
print(f"Saved: {OUTPUT_NPZ}  ({npz_mb:.1f} MB)")

## Step 3 — Verify the converted backbone

Load the `.npz` back, run a 224×224 forward pass, and inspect the logit statistics.
A freshly-pretrained backbone should produce non-trivial class distributions (not all
logits near zero) with a clear winning class for any given image.

In [ ]:
# Load the converted weights back and sanity-check the backbone.
darknet_v = Darknet(num_classes=1000)
darknet_v.load_weights(OUTPUT_NPZ)
darknet_v.eval = lambda: None   # Darknet has no eval() — BN always uses running stats

# Monkey-patch a silent eval forward (no per-layer prints)
import types

def _darknet_fwd(self, x):
    for layer in self.layers:
        x = layer.forward(x)
    x = self.gap.forward(x)
    return self.fc.forward(x)

darknet_v.forward = types.MethodType(_darknet_fwd, darknet_v)

# Forward pass on a batch of random 224×224 images.
x_dummy = cp.random.rand(4, 3, 224, 224, dtype=cp.float32)
logits  = darknet_v.forward(x_dummy)
logits_np = cp.asnumpy(logits)

print(f"Backbone forward pass: input {tuple(x_dummy.shape)} -> logits {tuple(logits_np.shape)}")
print(f"Logits stats: mean={logits_np.mean():+.4f}  std={logits_np.std():.4f}  "
      f"min={logits_np.min():+.4f}  max={logits_np.max():+.4f}")
print(f"NaN: {int(np.isnan(logits_np).sum())}   Inf: {int(np.isinf(logits_np).sum())}")

# Top-5 predicted class indices per image (ImageNet classes — just indices here)
for i in range(logits_np.shape[0]):
    top5 = np.argsort(logits_np[i])[::-1][:5]
    scores = logits_np[i][top5]
    print(f"  img {i}: top-5 class indices = {list(top5)}  logits = {[f'{s:.2f}' for s in scores]}")

assert not np.isnan(logits_np).any(), "NaN in backbone output — conversion failed."
assert logits_np.std() > 0.1, "Logit std near zero — weights not loaded correctly."
print("\nBackbone verification PASSED.")

## Step 4 — Insert backbone into YOLO

`Darknet.layers` (64 items) and `YOLO.backbone` (64 items) share the same
architecture, so the insertion is a direct 1-to-1 layer copy — identical to
`YOLO_backbone_insertion.ipynb`.

In [ ]:
from yolo import YOLO
from conv2d import Conv2D
from batchnorm2d import BatchNorm2D
from linear import Linear

yolo = YOLO(num_classes=20)

# Direct layer-for-layer copy: darknet_v.layers[i] -> yolo.backbone[i]
assert len(darknet_v.layers) == len(yolo.backbone), (
    f"Layer count mismatch: Darknet {len(darknet_v.layers)} vs YOLO backbone {len(yolo.backbone)}"
)
for i, src_layer in enumerate(darknet_v.layers):
    yolo.backbone[i] = src_layer

yolo.init_optimizer()

# Silent forward (suppress per-layer debug prints baked into yolo.py)
def _silent_forward(self, x):
    assert x.ndim == 4 and x.shape[1] == 3
    for layer in self.backbone:
        x = layer.forward(x)
    for layer in self.head:
        x = layer.forward(x)
    return x

yolo.forward = types.MethodType(_silent_forward, yolo)

# Pre-flight forward pass on a 448×448 dummy input
yolo.eval()
x_test = cp.random.rand(2, 3, 448, 448, dtype=cp.float32)
out    = yolo.forward(x_test)
print(f"YOLO forward pass: input {tuple(x_test.shape)} -> logits {tuple(out.shape)}")
assert out.shape == (2, 7 * 7 * 30), f"Unexpected output shape {out.shape}"
print("Backbone insertion PASSED.")

# Parameter summary
def _count_params(layer):
    n = 0
    if isinstance(layer, Conv2D):
        n += int(layer.weights.size)
        if layer.bias is not None:
            n += int(layer.bias.size)
    elif isinstance(layer, BatchNorm2D) and layer.affine:
        n += int(layer.gamma.size) + int(layer.beta.size)
    elif isinstance(layer, Linear):
        n += int(layer.W.size)
        if layer.use_bias:
            n += int(layer.b.size)
    return n

total_backbone = sum(_count_params(l) for l in yolo.backbone)
total_head     = sum(_count_params(l) for l in yolo.head)
print(f"\nBackbone params : {total_backbone:>14,}  (~{total_backbone/1e6:.2f} M)")
print(f"Head     params : {total_head:>14,}  (~{total_head/1e6:.2f} M)")
print(f"Total           : {total_backbone+total_head:>14,}  (~{(total_backbone+total_head)/1e6:.2f} M)")
print("Optimizer: SGD + momentum 0.9 + weight decay 5e-4  (velocity buffers allocated)")

---
## Training (identical to `YOLO_train.ipynb`)
All hyperparameters, LR schedule, loss, augmentation, and checkpoint logic below are
unchanged from the original training notebook.

In [ ]:
BATCH_SIZE   = 64
NUM_EPOCHS   = 60
MOMENTUM     = 0.9
WEIGHT_DECAY = 5e-4
LAMBDA_COORD = 5.0
LAMBDA_NOOBJ = 0.1
GRAD_ELEM_CLIP      = 10.0
MAX_PARAM_GRAD_NORM = 35.0

REPO         = "/content/yolov1-cupy"
VOC_DATA_ROOT = "/content/yolov1-cupy"
S, B, C      = 7, 2, 20
IMG_SIZE     = (448, 448)

In [ ]:
# LR schedule: compressed paper ratio, 60 epochs total.
# Epoch 0: warmup 1e-3 -> 1e-2 linearly over one epoch
# Epochs 1..34:  1e-2  (34 epochs)
# Epochs 35..49: 1e-3  (15 epochs)
# Epochs 50..59: 1e-4  (10 epochs)
def lr_at(epoch, step, steps_per_epoch):
    if epoch == 0:
        return 1e-3 + (1e-2 - 1e-3) * (step / max(steps_per_epoch, 1))
    if epoch < 35:
        return 1e-2
    if epoch < 50:
        return 1e-3
    return 1e-4

In [ ]:
import numpy as np
from image_batch_loader import voc_dataset_size, voc_num_batches_per_epoch

n_train_imgs    = voc_dataset_size(REPO, data_root=VOC_DATA_ROOT, split="train")
n_val_imgs      = voc_dataset_size(REPO, data_root=VOC_DATA_ROOT, split="val")
n_train_batches = voc_num_batches_per_epoch(REPO, BATCH_SIZE, data_root=VOC_DATA_ROOT, split="train")
n_val_batches   = voc_num_batches_per_epoch(REPO, BATCH_SIZE, data_root=VOC_DATA_ROOT, split="val")

print(f"Train: {n_train_imgs} images, {n_train_batches} batches/epoch")
print(f"Val:   {n_val_imgs} images, {n_val_batches} batches/epoch")
print(f"Total training steps: {NUM_EPOCHS * n_train_batches:,}")

In [ ]:
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from image_batch_loader import voc_image_target_batch_fast, VOC_CLASS_NAMES

# LR schedule preview
_lr_preview = np.array([
    lr_at(ep, st, n_train_batches)
    for ep in range(NUM_EPOCHS) for st in range(n_train_batches)
], dtype=np.float32)
fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(_lr_preview, color="tab:green", linewidth=1.5)
ax.set_yscale("log"); ax.set_xlabel("step"); ax.set_ylabel("LR")
ax.set_title(f"LR schedule ({NUM_EPOCHS} epochs × {n_train_batches} batches)")
ax.grid(alpha=0.3, which="both"); plt.tight_layout(); plt.show()

# Augmented batch preview
x_prev, y_prev = voc_image_target_batch_fast(
    REPO, batch_size=8, seed=0, batch_index=0,
    data_root=VOC_DATA_ROOT, split="train",
    size=IMG_SIZE, s=S, b=B, c=C, augment=True,
)
x_np = cp.asnumpy(x_prev); y_np = cp.asnumpy(y_prev)
H_img, W_img = IMG_SIZE[1], IMG_SIZE[0]

fig2, axes2 = plt.subplots(2, 4, figsize=(14, 7))
for i, ax in enumerate(axes2.flat):
    img = np.transpose(x_np[i], (1, 2, 0))
    ax.imshow(np.clip(img, 0.0, 1.0))
    ax.set_xticks([]); ax.set_yticks([])
    n_boxes = 0
    for gy in range(S):
        for gx in range(S):
            if y_np[i, gy, gx, 4] > 0.0:
                x_c = (gx + float(y_np[i, gy, gx, 0])) / S
                y_c = (gy + float(y_np[i, gy, gx, 1])) / S
                bw  = float(y_np[i, gy, gx, 2])
                bh  = float(y_np[i, gy, gx, 3])
                cls = int(np.argmax(y_np[i, gy, gx, B * 5:]))
                rect = mpatches.Rectangle(
                    ((x_c - bw/2)*W_img, (y_c - bh/2)*H_img), bw*W_img, bh*H_img,
                    linewidth=2, edgecolor="lime", facecolor="none")
                ax.add_patch(rect)
                ax.text((x_c-bw/2)*W_img, (y_c-bh/2)*H_img-4, VOC_CLASS_NAMES[cls],
                        color="black", fontsize=8,
                        bbox=dict(facecolor="lime", edgecolor="none", pad=1, alpha=0.8))
                n_boxes += 1
    ax.set_title(f"img {i}  ({n_boxes} boxes)", fontsize=9)
fig2.suptitle("Augmented training batch + GT boxes", fontsize=13)
fig2.tight_layout(); plt.show()

In [ ]:
# Live training-metrics dashboard (same as YOLO_train.ipynb)
from IPython.display import display

train_loss_per_batch = []
train_loss_ema       = []
lr_history           = []
epoch_boundary_step  = []
epoch_train_losses   = []
epoch_val_losses     = []
epoch_times          = []

_EMA_ALPHA = 0.02

fig_dash, axes_dash = plt.subplots(2, 2, figsize=(15, 9))
_dash_handle = None

PROGRESS_PNG_LOCAL = "/content/yolov1-cupy/models/training_progress_v2.png"
PROGRESS_PNG_DRIVE = f"{PATH_PREFIX}/checkpoints/training_progress_v2.png"


def render_dashboard(epoch_idx, total_epochs):
    for row in axes_dash:
        for ax in row:
            ax.clear()

    ax = axes_dash[0][0]
    ax.plot(train_loss_per_batch, color="tab:blue", alpha=0.25, label="per-batch")
    if len(train_loss_ema) == len(train_loss_per_batch):
        ax.plot(train_loss_ema, color="tab:blue", linewidth=2, label=f"EMA")
    for b in epoch_boundary_step:
        ax.axvline(b, color="gray", linestyle="--", alpha=0.3, linewidth=0.8)
    ax.set_xlabel("batch"); ax.set_ylabel("loss")
    ax.set_title(f"Train loss per batch (epoch {epoch_idx}/{total_epochs})")
    ax.legend(loc="upper right"); ax.grid(alpha=0.3)

    ax = axes_dash[0][1]
    es = list(range(1, len(epoch_train_losses) + 1))
    if es:
        ax.plot(es, epoch_train_losses, "o-", label="train", color="tab:blue")
        for x, y in zip(es, epoch_train_losses):
            ax.annotate(f"{y:.2f}", (x, y), textcoords="offset points", xytext=(0, 6), fontsize=7)
    if epoch_val_losses:
        es_v = list(range(1, len(epoch_val_losses) + 1))
        ax.plot(es_v, epoch_val_losses, "s-", label="val", color="tab:orange")
        for x, y in zip(es_v, epoch_val_losses):
            ax.annotate(f"{y:.2f}", (x, y), textcoords="offset points", xytext=(0, -12), fontsize=7, color="tab:orange")
    ax.set_xlabel("epoch"); ax.set_ylabel("avg loss")
    ax.set_title("Epoch average loss"); ax.legend(); ax.grid(alpha=0.3)

    ax = axes_dash[1][0]
    ax.plot(lr_history, color="tab:green", linewidth=1.2)
    for b in epoch_boundary_step:
        ax.axvline(b, color="gray", linestyle="--", alpha=0.3, linewidth=0.8)
    ax.set_xlabel("batch"); ax.set_ylabel("LR")
    ax.set_title("Learning rate schedule (actual)")
    ax.set_yscale("log"); ax.grid(alpha=0.3, which="both")

    ax = axes_dash[1][1]
    if epoch_times:
        xs = list(range(1, len(epoch_times) + 1))
        ax.bar(xs, epoch_times, color="tab:purple", alpha=0.8)
        mean_t = sum(epoch_times) / len(epoch_times)
        ax.axhline(mean_t, color="black", linestyle="--", linewidth=1, label=f"mean={mean_t:.1f}s")
        ax.legend()
    ax.set_xlabel("epoch"); ax.set_ylabel("seconds")
    ax.set_title("Wall time per epoch"); ax.grid(alpha=0.3, axis="y")

    fig_dash.suptitle(f"YOLOv1 v2 training  —  epoch {epoch_idx}/{total_epochs}", fontsize=14)
    fig_dash.tight_layout(rect=[0, 0, 1, 0.96])

    try:
        fig_dash.savefig(PROGRESS_PNG_LOCAL, dpi=100)
        shutil.copy2(PROGRESS_PNG_LOCAL, PROGRESS_PNG_DRIVE)
    except Exception as exc:
        print(f"  [warn] failed to save dashboard: {exc}")

    global _dash_handle
    if _dash_handle is None:
        _dash_handle = display(fig_dash, display_id=True)
    else:
        _dash_handle.update(fig_dash)


print("Dashboard ready.")

In [ ]:
# =============================================================================
# Pre-flight NaN/Inf diagnostic — identical to YOLO_train.ipynb.
# =============================================================================
from loss import yolo_loss, yolo_loss_grad


def _stats(tag, arr):
    a = cp.asnumpy(arr) if isinstance(arr, cp.ndarray) else np.asarray(arr)
    nan_n = int(np.isnan(a).sum())
    inf_n = int(np.isinf(a).sum())
    flag = "  OK " if (nan_n == 0 and inf_n == 0) else " BAD "
    print(f"  [{flag}] {tag:<36s}  shape={str(a.shape):<22s} "
          f"mean={a.mean():>+9.3e}  std={a.std():>9.3e}  NaN={nan_n}  Inf={inf_n}")
    return nan_n, inf_n


def _run_diag(augment):
    label = f" diag: augment={augment} "
    print("-" * 72)
    print(label.center(72, "-"))
    print("-" * 72)

    x, y = voc_image_target_batch_fast(
        REPO, batch_size=BATCH_SIZE, seed=0, batch_index=0,
        data_root=VOC_DATA_ROOT, split="train",
        size=IMG_SIZE, s=S, b=B, c=C, augment=augment,
    )
    _stats("input x", x); _stats("target y", y)

    if augment:
        yolo.train()
    else:
        yolo.eval()

    logits = yolo.forward(x)
    n1, i1 = _stats("logits", logits)
    if n1 or i1:
        print("  --> NaN/Inf in logits. STOP."); return False

    loss = yolo_loss(logits, y, S=S, B=B, C=C,
                     lambda_coord=LAMBDA_COORD, lambda_noobj=LAMBDA_NOOBJ)
    print(f"  loss value : {float(loss):.6f}  (finite={np.isfinite(float(loss))})")
    if not np.isfinite(float(loss)):
        print("  --> loss is NaN/Inf. STOP."); return False

    grad = yolo_loss_grad(logits, y, S=S, B=B, C=C,
                          lambda_coord=LAMBDA_COORD, lambda_noobj=LAMBDA_NOOBJ)
    n1, i1 = _stats("grad wrt logits", grad)
    if n1 or i1:
        print("  --> NaN/Inf in grad. STOP."); return False

    yolo.zero_grad()
    yolo.backward(grad)

    max_rows = []
    for section, layers in (("backbone", yolo.backbone), ("head", yolo.head)):
        for lyr in layers:
            for attr in ("dW", "dgamma"):
                g = getattr(lyr, attr, None)
                if g is not None:
                    a = cp.asnumpy(g)
                    max_rows.append((float(np.abs(a).max()), f"{section}:{type(lyr).__name__}",
                                     int(np.isnan(a).sum()), int(np.isinf(a).sum())))
    print("  ---- |dW|.max() per trainable layer (top 8) ----")
    for mx, name, nan_n, inf_n in sorted(max_rows, reverse=True)[:8]:
        print(f"     {name:<30s}  max|dW|={mx:>9.3e}   NaN={nan_n}  Inf={inf_n}")
    if any(r[2] or r[3] for r in max_rows):
        print("  --> NaN/Inf in dW. STOP."); return False
    print("  diag passed.")
    return True


ok1 = _run_diag(augment=False)
ok2 = _run_diag(augment=True)
print("=" * 72)
if ok1 and ok2:
    print("  PRE-FLIGHT OK — safe to train.")
else:
    print("  PRE-FLIGHT FAILED — fix the BAD row above before training.")
print("=" * 72)

In [ ]:
# =============================================================================
# Training + validation loop — identical to YOLO_train.ipynb.
# =============================================================================
import time
from tqdm.auto import tqdm
from image_batch_loader import BatchPrefetcher

_GRAD_ATTRS = ("dW", "db", "dgamma", "dbeta")


def _iter_param_grads(model):
    for layer in list(model.backbone) + list(model.head):
        for attr in _GRAD_ATTRS:
            g = getattr(layer, attr, None)
            if g is not None:
                yield g


def clip_parameter_grad_norm(model, max_norm):
    total_sq = cp.zeros((), dtype=cp.float32)
    for g in _iter_param_grads(model):
        total_sq += (g * g).sum(dtype=cp.float32)
    total_norm = float(cp.sqrt(total_sq))
    if total_norm > max_norm and total_norm > 0.0:
        scale = max_norm / (total_norm + 1e-8)
        for g in _iter_param_grads(model):
            g *= scale
    return total_norm


def _log_line(line):
    print(line)
    with open(LOCAL_LOG_PATH, "a") as fh:
        fh.write(line + "\n")
    try:
        shutil.copy2(LOCAL_LOG_PATH, DRIVE_LOG_PATH)
    except Exception:
        pass


def _fmt_seconds(s):
    s = int(s)
    h, r = divmod(s, 3600)
    m, s = divmod(r, 60)
    return f"{h:d}h{m:02d}m{s:02d}s" if h else (f"{m:d}m{s:02d}s" if m else f"{s:d}s")


print("#" * 64)
print(f"# Starting YOLOv1 v2 training: {NUM_EPOCHS} epochs x {n_train_batches} batches")
print(f"# Drive checkpoints -> {DRIVE_CKPT_DIR}")
print("#" * 64)

best_val         = float("inf")
training_started = time.time()

for epoch in range(NUM_EPOCHS):
    epoch_started = time.time()

    # -- Training phase --
    yolo.train()
    train_loss_sum   = 0.0
    train_loss_count = 0
    running_ema      = None
    imgs_seen        = 0

    train_pref = BatchPrefetcher(
        REPO, BATCH_SIZE, seed=epoch, data_root=VOC_DATA_ROOT, split="train",
        n_batches=n_train_batches, size=IMG_SIZE, s=S, b=B, c=C, augment=True,
    )
    pbar = tqdm(train_pref, total=n_train_batches,
                desc=f"[train {epoch+1:02d}/{NUM_EPOCHS:02d}]", leave=True, dynamic_ncols=True)
    try:
        for step, (x, y) in enumerate(pbar):
            lr = lr_at(epoch, step, n_train_batches)
            yolo.zero_grad()
            logits = yolo.forward(x)
            loss   = yolo_loss(logits, y, S=S, B=B, C=C,
                               lambda_coord=LAMBDA_COORD, lambda_noobj=LAMBDA_NOOBJ)
            grad   = yolo_loss_grad(logits, y, S=S, B=B, C=C,
                                    lambda_coord=LAMBDA_COORD, lambda_noobj=LAMBDA_NOOBJ)
            batch_n = float(x.shape[0])
            grad    = grad / batch_n
            cp.clip(grad, -GRAD_ELEM_CLIP, GRAD_ELEM_CLIP, out=grad)
            yolo.backward(grad)
            gnorm = clip_parameter_grad_norm(yolo, MAX_PARAM_GRAD_NORM)
            yolo.sgd_momentum_step(lr, MOMENTUM, WEIGHT_DECAY)

            loss_f = float(loss) / batch_n
            if not np.isfinite(loss_f):
                raise RuntimeError(f"loss became {loss_f} at epoch {epoch+1} step {step}")
            train_loss_sum   += loss_f
            train_loss_count += 1
            imgs_seen        += int(x.shape[0])

            running_ema = loss_f if running_ema is None else (
                (1 - _EMA_ALPHA) * running_ema + _EMA_ALPHA * loss_f
            )
            train_loss_per_batch.append(loss_f)
            train_loss_ema.append(running_ema)
            lr_history.append(lr)

            elapsed = max(time.time() - epoch_started, 1e-6)
            pbar.set_postfix({
                "loss":  f"{loss_f:.3f}",
                "ema":   f"{running_ema:.3f}",
                "avg":   f"{train_loss_sum/train_loss_count:.3f}",
                "lr":    f"{lr:.1e}",
                "gnorm": f"{gnorm:.2f}",
                "img/s": f"{imgs_seen/elapsed:5.1f}",
            })
    finally:
        pbar.close()
        train_pref.close()

    train_avg = train_loss_sum / max(train_loss_count, 1)
    epoch_boundary_step.append(len(train_loss_per_batch))
    epoch_train_losses.append(train_avg)

    # -- Validation phase --
    yolo.eval()
    val_loss_sum   = 0.0
    val_loss_count = 0
    val_pref = BatchPrefetcher(
        REPO, BATCH_SIZE, seed=0, data_root=VOC_DATA_ROOT, split="val",
        n_batches=n_val_batches, size=IMG_SIZE, s=S, b=B, c=C, augment=False,
    )
    vbar = tqdm(val_pref, total=n_val_batches,
                desc=f"[ val  {epoch+1:02d}/{NUM_EPOCHS:02d}]", leave=True, dynamic_ncols=True)
    try:
        for x, y in vbar:
            logits_v = yolo.forward(x)
            vloss = float(yolo_loss(
                logits_v, y, S=S, B=B, C=C,
                lambda_coord=LAMBDA_COORD, lambda_noobj=LAMBDA_NOOBJ,
            )) / float(x.shape[0])
            val_loss_sum   += vloss
            val_loss_count += 1
            vbar.set_postfix({"loss": f"{vloss:.3f}",
                              "avg":  f"{val_loss_sum/val_loss_count:.3f}"})
    finally:
        vbar.close()
        val_pref.close()

    val_avg = val_loss_sum / max(val_loss_count, 1)
    epoch_val_losses.append(val_avg)
    epoch_elapsed = time.time() - epoch_started
    epoch_times.append(epoch_elapsed)

    is_best = val_avg < best_val
    if is_best:
        best_val = val_avg

    lr_start  = lr_at(epoch, 0, n_train_batches)
    lr_end    = lr_at(epoch, n_train_batches - 1, n_train_batches)
    total_el  = time.time() - training_started
    remaining = epoch_elapsed * (NUM_EPOCHS - epoch - 1)

    print()
    print("+" + "-" * 62 + "+")
    print(f"|  epoch {epoch+1:>3d}/{NUM_EPOCHS:<3d}  finished in {_fmt_seconds(epoch_elapsed):<12s}" + " " * 14 + "|")
    print("+" + "-" * 62 + "+")
    print(f"|  train loss : {train_avg:>10.4f}" + " " * 36 + "|")
    best_marker = "  <-- NEW BEST" if is_best else ""
    print(f"|  val   loss : {val_avg:>10.4f}   best : {best_val:>10.4f}{best_marker:<13s}|")
    print(f"|  LR         : start {lr_start:.1e}  end {lr_end:.1e}" + " " * 16 + "|")
    print(f"|  elapsed    : {_fmt_seconds(total_el):<12s}   ETA : {_fmt_seconds(remaining):<14s}" + " " * 10 + "|")
    print("+" + "-" * 62 + "+")

    _log_line(
        f"epoch {epoch+1:03d}/{NUM_EPOCHS} "
        f"train={train_avg:.4f} val={val_avg:.4f} best={best_val:.4f} "
        f"lr_start={lr_start:.1e} lr_end={lr_end:.1e} "
        f"time={epoch_elapsed:.1f}s elapsed={_fmt_seconds(total_el)} eta={_fmt_seconds(remaining)}"
    )

    render_dashboard(epoch + 1, NUM_EPOCHS)

    local_ckpt = os.path.join(LOCAL_CKPT_DIR, f"yolo-v2-epoch{epoch+1}.npz")
    drive_ckpt = os.path.join(DRIVE_CKPT_DIR, f"yolo-v2-epoch{epoch+1}.npz")
    yolo.save_weights(local_ckpt)
    shutil.copy2(local_ckpt, drive_ckpt)
    print(f"  saved: {local_ckpt}")
    if is_best:
        best_local_ckpt = os.path.join(LOCAL_CKPT_DIR, "yolo-v2-best.npz")
        best_drive_ckpt = os.path.join(DRIVE_CKPT_DIR, "yolo-v2-best.npz")
        shutil.copy2(local_ckpt, best_local_ckpt)
        shutil.copy2(best_local_ckpt, best_drive_ckpt)
        print(f"  best updated: {best_drive_ckpt}")

print()
print("#" * 64)
print(f"# Training complete in {_fmt_seconds(time.time() - training_started)}")
print(f"# Best val loss: {best_val:.4f}")
print("#" * 64)

In [ ]:
# =============================================================================
# Post-training sanity check — identical to YOLO_train.ipynb.
# =============================================================================
print("=" * 72)
print("POST-TRAINING SANITY CHECK")
print("=" * 72)

# 1. Checkpoint inventory
print("\n[1] Checkpoint inventory")
missing_local = []
missing_drive = []
sizes = []
for ep in range(1, NUM_EPOCHS + 1):
    lp = os.path.join(LOCAL_CKPT_DIR, f"yolo-v2-epoch{ep}.npz")
    dp = os.path.join(DRIVE_CKPT_DIR, f"yolo-v2-epoch{ep}.npz")
    if os.path.isfile(lp):
        sizes.append(os.path.getsize(lp))
    else:
        missing_local.append(ep)
    if not os.path.isfile(dp):
        missing_drive.append(ep)

n_local = NUM_EPOCHS - len(missing_local)
n_drive = NUM_EPOCHS - len(missing_drive)
mean_mb = (sum(sizes)/len(sizes))/2**20 if sizes else 0.0
print(f"  local : {n_local}/{NUM_EPOCHS} found  (avg {mean_mb:.1f} MB)")
print(f"  drive : {n_drive}/{NUM_EPOCHS} found")
best_local = os.path.join(LOCAL_CKPT_DIR, "yolo-v2-best.npz")
best_drive = os.path.join(DRIVE_CKPT_DIR, "yolo-v2-best.npz")
print(f"  best  : local={'OK' if os.path.isfile(best_local) else 'MISSING'}  "
      f"drive={'OK' if os.path.isfile(best_drive) else 'MISSING'}")
if missing_local:
    print(f"  [warn] missing locally: {missing_local}")

# 2. Log file
print("\n[2] Training log")
if os.path.isfile(LOCAL_LOG_PATH):
    with open(LOCAL_LOG_PATH) as fh:
        lines = fh.readlines()
    print(f"  {len(lines)} lines")
    for line in lines[-3:]:
        print(f"    {line.rstrip()}")

# 3. Round-trip
print("\n[3] Round-trip test")
reload_path = best_local if os.path.isfile(best_local) else os.path.join(
    LOCAL_CKPT_DIR, f"yolo-v2-epoch{NUM_EPOCHS}.npz")
_rt = YOLO(num_classes=20)
_rt.load_weights(reload_path)
_rt.eval()
_rt.forward = types.MethodType(_silent_forward, _rt)
tmp = "/content/yolov1-cupy/models/_rt_v2.npz"
_rt.save_weights(tmp)
_rt2 = YOLO(num_classes=20)
_rt2.load_weights(tmp)
_rt2.forward = types.MethodType(_silent_forward, _rt2)
nan_count = inf_count = 0
ok_rt = True
for a, b_ in zip(list(_rt.backbone)+list(_rt.head), list(_rt2.backbone)+list(_rt2.head)):
    if isinstance(a, Conv2D):
        nan_count += int(cp.isnan(a.weights).sum().get())
        if not cp.allclose(a.weights, b_.weights): ok_rt = False
    elif isinstance(a, BatchNorm2D) and a.affine:
        if not cp.allclose(a.gamma, b_.gamma): ok_rt = False
    elif isinstance(a, Linear):
        nan_count += int(cp.isnan(a.W).sum().get())
        if not cp.allclose(a.W, b_.W): ok_rt = False
print(f"  round-trip match: {'OK' if ok_rt else 'FAIL'}  NaN params: {nan_count}")

# 4. Forward pass
print("\n[4] Forward pass (eval mode, val batch)")
x_t, y_t = voc_image_target_batch_fast(
    REPO, batch_size=8, seed=0, batch_index=0,
    data_root=VOC_DATA_ROOT, split="val",
    size=IMG_SIZE, s=S, b=B, c=C, augment=False,
)
t0 = time.time()
logits_t = _rt2.forward(x_t)
cp.cuda.Stream.null.synchronize()
fwd_ms = (time.time() - t0) * 1000
logits_np = cp.asnumpy(logits_t)
has_nan = bool(np.isnan(logits_np).any())
has_inf = bool(np.isinf(logits_np).any())
vl_here = float(yolo_loss(logits_t, y_t, S=S, B=B, C=C,
                           lambda_coord=LAMBDA_COORD, lambda_noobj=LAMBDA_NOOBJ)) / 8.0
print(f"  forward {fwd_ms:.1f} ms  logits mean={logits_np.mean():.4f} std={logits_np.std():.4f}  "
      f"NaN={has_nan} Inf={has_inf}  val_loss={vl_here:.4f}")

# 5. Training curve
print("\n[5] Training curve")
if epoch_train_losses:
    print(f"  train: first={epoch_train_losses[0]:.4f}  last={epoch_train_losses[-1]:.4f}  "
          f"delta={epoch_train_losses[0]-epoch_train_losses[-1]:+.4f}")
    print(f"  val:   first={epoch_val_losses[0]:.4f}  last={epoch_val_losses[-1]:.4f}  "
          f"best={best_val:.4f}")

# Verdict
ok_all = (n_local == NUM_EPOCHS and not has_nan and not has_inf
          and ok_rt and nan_count == 0
          and epoch_train_losses and epoch_train_losses[0] > epoch_train_losses[-1])
print("\n" + "=" * 72)
print("ALL CHECKS PASSED" if ok_all else "SOME CHECKS FAILED — see warnings above")
print("=" * 72)

In [ ]:
# =============================================================================
# Save final artifacts to Drive.
# =============================================================================
import json

FINAL_DIR_LOCAL = "/content/yolov1-cupy/models/final_v2"
FINAL_DIR_DRIVE = f"{PATH_PREFIX}/checkpoints/final_v2"
os.makedirs(FINAL_DIR_LOCAL, exist_ok=True)
os.makedirs(FINAL_DIR_DRIVE, exist_ok=True)

render_dashboard(len(epoch_train_losses), NUM_EPOCHS)
dash_local = os.path.join(FINAL_DIR_LOCAL, "training_progress.png")
dash_drive = os.path.join(FINAL_DIR_DRIVE, "training_progress.png")
fig_dash.savefig(dash_local, dpi=120, bbox_inches="tight")
shutil.copy2(dash_local, dash_drive)

metrics = {
    "version": "v2",
    "backbone": "official darknet ImageNet-1K (pjreddie.com/media/files/darknet.weights)",
    "hyperparameters": {
        "batch_size": BATCH_SIZE, "num_epochs": NUM_EPOCHS,
        "momentum": MOMENTUM, "weight_decay": WEIGHT_DECAY,
        "lambda_coord": LAMBDA_COORD, "lambda_noobj": LAMBDA_NOOBJ,
        "image_size": list(IMG_SIZE), "S": S, "B": B, "C": C,
    },
    "dataset": {
        "train_images": n_train_imgs, "val_images": n_val_imgs,
        "train_batches_per_epoch": n_train_batches, "val_batches_per_epoch": n_val_batches,
    },
    "results": {
        "epochs_completed": len(epoch_train_losses),
        "best_val_loss": float(best_val),
        "train_loss_per_epoch": [float(v) for v in epoch_train_losses],
        "val_loss_per_epoch":   [float(v) for v in epoch_val_losses],
        "epoch_time_sec":       [float(v) for v in epoch_times],
    },
}
metrics_local = os.path.join(FINAL_DIR_LOCAL, "metrics.json")
metrics_drive = os.path.join(FINAL_DIR_DRIVE, "metrics.json")
with open(metrics_local, "w") as fh:
    json.dump(metrics, fh, indent=2)
shutil.copy2(metrics_local, metrics_drive)

print("=" * 72)
print("Final artifacts saved:")
print(f"  local : {FINAL_DIR_LOCAL}/")
print(f"  drive : {FINAL_DIR_DRIVE}/")
print(f"  best val loss : {best_val:.4f}")
print(f"  epochs        : {len(epoch_train_losses)}/{NUM_EPOCHS}")
if epoch_times:
    print(f"  total time    : {_fmt_seconds(sum(epoch_times))}")
print("=" * 72)
print("\nNext: run YOLO_test.ipynb pointing to yolo-v2-best.npz to measure mAP@0.5.")